# 12. BIEs I: Introduction and Nystrom discretization

Boundary integral equation (BIE) methods reduce certain problems for partial differential equations to equations for functions defined on a boundary. Before looking at boundary integrals, we will study the basic structure of integral equations, and a numerical method for discretizing them: the **Nystrom method**.

We will see that an integral equation can be viewed much like a linear system, except that its unknown is a function. We will then turn it into a finite-dimensional linear system by approximating an integral with a quadrature rule.

## 12.1 Integral equations and integral operators

Suppose we are given an interval $[a,b]$, a function $f:[a,b]\to\mathbb{R}$, and a **kernel** $k:[a,b]\times[a,b]\to\mathbb{R}$. We would like to find an unknown function $u$ satisfying

$$
\int_a^b k(t,s)u(s)\,ds=f(t), \qquad t\in[a,b].
$$

This is a **Fredholm integral equation of the first kind**. The integration variable is $s$; the equation must hold for every $t$ in the interval. Notice that we are solving for the entire function $u$, not just for a finite number of values.

A **Fredholm integral equation of the second kind** has an additional term involving $u$ itself:

$$
u(t)-\int_a^b k(t,s)u(s)\,ds=f(t), \qquad t\in[a,b].
$$

The two kinds of equations have very different properties, and for numerical solution, second kind Fredholm equations are preferred. In particular, first-kind equations can have nonunique solutions (leading to poor conditioning), as we will see in an example.

### 12.1.1 Integral operators

Define the integral operator $K$ by

$$
(Ku)(t)=\int_a^b k(t,s)u(s)\,ds.
$$

Then the first-kind equation is simply

$$
Ku=f,
$$

while the second-kind equation is

$$
(I-K)u=f,
$$

where $I$ is the identity operator, $(Iu)(t)=u(t)$.

We can think of $K$ as an infinite-dimensional analogue of a matrix: $u$ is the unknown *function*, $f$ is the right-hand side, and $k(t,s)$ plays the role of a matrix entry indexed by two continuous variables. If we fix $t$, then $k(t,\cdot)$ is analogous to one row of a matrix.

```{admonition} Question
What is $K^2u$? 
```
Just as with a matrix, $K^2$ means applying $K$ twice:

$$
\begin{aligned}
(K^2u)(t)
&=\int_a^b k(t,s)(Ku)(s)\,ds\\
&=\int_a^b k(t,s)\left[\int_a^b k(s,r)u(r)\,dr\right]ds\\
&=\int_a^b\left[\int_a^b k(t,s)k(s,r)\,ds\right]u(r)\,dr.
\end{aligned}
$$

Assuming we can exchange the order of integration, $K^2$ is itself an integral operator, with kernel

$$
k^{(2)}(t,r)=\int_a^b k(t,s)k(s,r)\,ds.
$$

Compare this with matrix multiplication: $(A^2)_{ij}=\sum_\ell A_{i\ell}A_{\ell j}$. 

## 12.2 An example: nonuniqueness of a first-kind equation

Consider

$$
\int_0^1 t^2s\,u(s)\,ds=\frac{t^2}{3},\qquad 0<t<1.
$$

Because $t^2$ is constant with respect to the integration variable $s$, we can factor it out:

$$
t^2\int_0^1 s\,u(s)\,ds=\frac{t^2}{3}.
$$

For $t>0$ the equation is equivalent to one scalar constraint on the unknown function,

$$
\int_0^1 s\,u(s)\,ds=\frac13.
$$

One solution is $u(s)=s$, since $\int_0^1s^2\,ds=1/3$. But $u(s)=2/3$ is also a solution. In fact, so is every function of the form

$$
u(s)=s+v(s), \qquad \text{where }\int_0^1 s\,v(s)\,ds=0,
$$

i.e. for any $v$ orthogonal to $s$ on the interval $[0, 1]$ (their inner product being zero). The operator here has rank one:

$$
(Ku)(t)=t^2\left(\int_0^1s\,u(s)\,ds\right).
$$

No matter what $u$ we put into $K$, the output is a multiple of $t^2$. Thus the equation cannot determine the component of $u$ that is invisible to this integral. This is one reason why first-kind integral equations require particular care.

## 12.3 Bounded integral operators

To discuss the size of an integral operator, we need to choose norms on the functions we are working with. Given an operator $K$ between normed spaces, its induced (operator) norm is

$$
\|K\|=\sup_{\|u\|=1}\|Ku\|.
$$

We will use the space $C([a,b])$ of continuous functions on $[a,b]$, with the supremum norm

$$
\|u\|_\infty=\max_{t\in[a,b]}|u(t)|.
$$

A linear operator is **bounded** if its operator norm is finite.

### 12.3.1 The supremum norm of an integral operator

Assume first that the kernel $k$ is continuous on $[a,b]^2$. If $\|u\|_\infty=1$, then for any $t\in[a,b]$,

$$
\begin{aligned}
|(Ku)(t)|
&=\left|\int_a^b k(t,s)u(s)\,ds\right|\\
&\leq\int_a^b |k(t,s)|\,|u(s)|\,ds\\
&\leq\int_a^b |k(t,s)|\,ds.
\end{aligned}
$$

Taking the maximum over $t$ and then the supremum over $u$ gives

$$
\|K\|_\infty\leq\max_{t\in[a,b]}\int_a^b |k(t,s)|\,ds.
$$

For continuous kernels, this inequality is actually an equality:

$$
\|K\|_\infty=\max_{t\in[a,b]}\int_a^b |k(t,s)|\,ds.
$$

To see why the bound is sharp, choose $t_0$ that maximizes the integral. We can choose a continuous function $u(s)$ of size at most $1$ that approximates the sign of $k(t_0,s)$ as closely as we like. Then $k(t_0,s)u(s)$ is approximately $|k(t_0,s)|$, so $|(Ku)(t_0)|$ can be made arbitrarily close to the right-hand side. (For a complex-valued kernel, use an approximation to its conjugate phase instead of its sign.)

In matrix terminology, the operator norm is the largest absolute row sum, but with the sum replaced by an integral.

### 12.3.2 Weakly singular kernels

A kernel does not have to be bounded pointwise for its integral operator to be bounded. For example, suppose that for $t\neq s$,

$$
|k(t,s)|\leq \frac{C}{|t-s|^\delta},\qquad 0<\delta<1.
$$

The kernel can blow up on the diagonal $t=s$. However, the singularity is **integrable**:

$$
\begin{aligned}
\int_a^b |k(t,s)|\,ds
&\leq C\int_a^b\frac{ds}{|t-s|^\delta}\\
&=\frac{C}{1-\delta}\big[(t-a)^{1-\delta}+(b-t)^{1-\delta}\big]\\
&\leq\frac{C\,2^\delta}{1-\delta}(b-a)^{1-\delta}.
\end{aligned}
$$

Thus the integral of the absolute value of each row remains uniformly bounded, and $K$ is a bounded operator on $C([a,b])$ when it maps continuous functions into continuous functions, as it does for the standard weakly singular kernels of this form.

Such kernels are called **weakly singular**, ant they are common in boundary integral equations. The basic Nystrom discretization below is most straightforward for a continuous kernel; a singularity on the diagonal requires additional care when evaluating quadrature terms. Sometimes the limit of the kernel as one approaches the diagonal is finite (we will see an example for that later) and can be evaluated analytically, otherwise it requires singular quadrature.

## 12.4 Nystrom discretization

We now turn to a numerical method for solving a **second-kind Fredholm equation**,

$$
u(t)-\int_a^b k(t,s)u(s)\,ds=f(t),\qquad t\in[a,b],
$$

or, in operator notation, $(I-K)u=f$.

The key idea is to approximate twice: first, the integral is replaced with a quadrature rule, then, the equation is evaluated at discrete nodes. Let $s_1,\ldots,s_n$ and $w_1,\ldots,w_n$ be quadrature nodes and weights, so that

$$
\int_a^b g(s)\,ds\approx\sum_{j=1}^n w_j g(s_j).
$$

At each fixed $t$, we apply this rule to the integrand $g(s)=k(t,s)u(s)$.

### 12.4.1 Approximating the operator

Define the approximate operator $K_n$ by

$$
(K_n u)(t)=\sum_{j=1}^n w_j k(t,s_j)u(s_j).
$$

The Nystrom approximation $u_n$ is defined as the solution of

$$
u_n(t)-\sum_{j=1}^n w_j k(t,s_j)u_n(s_j)=f(t), \qquad t\in[a,b].
$$

This is still an equation for a function $u_n(t)$, but its right-hand side depends on only $n$ unknown values: $u_n(s_1),\ldots,u_n(s_n)$.

Notice that $K_n$ is a **finite-rank** operator. Its range is contained in the span of the $n$ functions $k(\cdot,s_j)$:

$$
\operatorname{range}(K_n)\subseteq
\operatorname{span}\{k(\cdot,s_1),\ldots,k(\cdot,s_n)\}.
$$

In other words, $K_nu$ is a linear combination of $n$ column slices of the kernel, each obtained by keeping $s=s_j$ fixed.

### 12.4.2 The linear system at the quadrature nodes

We now choose $t=s_i$, for each $i=1,\ldots,n$. Denote the unknown nodal values by

$$
u_i^{(n)}:=u_n(s_i).
$$

The function equation becomes

$$
u_i^{(n)}-\sum_{j=1}^n w_j k(s_i,s_j)u_j^{(n)}=f(s_i),
\qquad i=1,\ldots,n.
$$

Define the $n\times n$ matrix $A$ by

$$
A_{ij}=w_j k(s_i,s_j),
$$

and the vectors

$$
\mathbf{u}^{(n)}=(u_1^{(n)},\ldots,u_n^{(n)})^T,
\qquad \mathbf{f}=(f(s_1),\ldots,f(s_n))^T.
$$

Then the discrete problem is the linear system

$$
(I-A)\mathbf{u}^{(n)}=\mathbf{f}.
$$

The entries of $A$ are obtained directly by evaluating the kernel at pairs of nodes and multiplying each *column* by the corresponding quadrature weight. Once we solve this system, we know the approximate solution at every quadrature node.

### 12.4.3 Reconstructing the function: Nystrom interpolation

Finally, we would like to evaluate $u_n(t)$ at an arbitrary point $t$, not just at the quadrature nodes. We could interpolate the values $u_i^{(n)}$ with a polynomial, but the original discretized equation gives us a more natural formula.

Rearranging $(I-K_n)u_n=f$ yields

$$
u_n(t)=f(t)+\sum_{j=1}^n w_j k(t,s_j)u_j^{(n)}.
$$

Surprisingly, this formula is **exact for the Nystrom approximation $u_n$.** We are not attempting to interpolate the unknown exact solution $u$ directly.

To prove exactness, substitute $t=s_i$. The linear system tells us that

$$
f(s_i)+\sum_{j=1}^n w_jk(s_i,s_j)u_j^{(n)}=u_i^{(n)}.
$$

Thus the reconstructed function agrees with the nodal solution. Conversely, given any solution of the linear system, the formula defines a function that satisfies the approximate equation for all $t\in[a,b]$.

The construction is sometimes called **Nystrom interpolation**. Where Lagrange polynomial interpolation would use the node polynomials as the basis for the interpolant, Nystrom uses the kernel functions $k(t,s_j)$ as a basis, together with $f(t)$.

### 12.4.4 A note on convergence

How accurate is our discretization of the kernel, and what does this mean for the accuracy of $u_n(t)$? 

We cannot claim that $\|K_n-K\|_\infty\to0$ as $n\to\infty$. For each $n$, the quadrature samples only finitely many values of $u$, while the true integral depends on the function over the whole interval; the operator norm allows us to choose a different, highly oscillatory function for every $n$.

The useful convergence statement is instead that, under the usual assumptions of a continuous kernel and a convergent quadrature rule, for every *fixed* function $u\in C([a,b])$,

$$
\|(K_n-K)u\|_\infty\longrightarrow0,\qquad n\to\infty.
$$

This is pointwise convergence of operators, not convergence in operator norm. Convergence of the solutions $u_n$ to $u$ also requires that the second-kind problem is well posed and that the discretized equations are stable.